# All steps: one visit and a dataset from T1w (+ T2w) to MNI
The whole pipeline of `Script/neonorm.py`. Steps 1 and 2 are those of `1.quickstart_direct_normalize.ipynb`: **run it first** (same example data, same `DATA_ROOT`, its review tables). This notebook adds **step 3**, the visit to MNI by each way, for one visit and for the dataset, then how to use the outputs.

| step | function | parts | main output (`<DATA_ROOT>/<id>/`) |
|---|---|---|---|
| 1 | `preprocess` | 1a orientation, 1b skull stripping + rigid T2, 1c QC | `<id>_T1w_brain_mask.nii.gz`, `<id>_preprocess.json` |
| 2 | `direct_normalize` | 2a skull stripping + N4, 2b registration to `direct_templates(age)`, 2c QC | `<id>_direct_to-<template>_*`, `<id>_direct_normalize.json`, `_qc.json` |
| 3 | `visit_to_mni` | 3a the transforms of a way composed into one field, 3b the images moved with it + log Jacobian, 3c metrics | `<id>_<way>[-<path>]_to-MNI152NLin2009cSym_{T1w,cerebellum_dseg,brain_mask,logjd}.nii.gz`, `.json` |

Which way to use for which age (direct, tpl2tpl, avgsubj) follows from the evaluation (`Notebook/5_evaluation.ipynb`; **not decided yet**, section 5). The project notebooks do the same for BCP and HCPD in more detail: 1 (dataset table), 2 (skull stripping), 3 (direct normalisation and its review), 4 (template edges, visits to MNI), 5 (evaluation). All options: `README.md`.

## 0. Setup
Once per computer: `bash Script/setup_neonorm.sh <software folder> <templates source>` (README, Installation; FSL 6.0.6.2 and FreeSurfer 8.0.0 set up first). It creates the conda environment `neonorm`, downloads ANTs 2.6.2, puts the SynthStrip models in one folder, copies and checks the templates, and writes the settings of the computer to `<software folder>/neonorm_env.sh`. Then, every time, `source <software folder>/neonorm_env.sh` in the shell **before** starting Jupyter, and the same line as `SETUP` of the Slurm job scripts. The cell below lists what is missing (empty: ready).

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO = Path('../..').resolve()  # this notebook is in <repository>/docs/tutorials
sys.path.insert(0, str(REPO / 'Script'))
os.environ.setdefault('NEONORM_BCP_TEMPLATES', str(REPO / 'templates' / 'BCP-atlas-for_release-Ver2.0.0'))
os.environ.setdefault('NEONORM_MNI_TEMPLATE', str(REPO / 'templates' / 'tpl-MNI152NLin2009cSym'))
os.environ.setdefault('NEONORM_NTHREADS', '8')  # CPU threads of ANTs in this notebook
import neonorm

SETUP = 'source /path/to/software/neonorm_env.sh'  # the setup line of every Slurm job script (setup_neonorm.sh)
EXAMPLE = Path('example').resolve()                 # absolute: the jsons record the paths
DATA_ROOT = EXAMPLE / 'out'                         # outputs, one folder per visit; review tables, Slurm logs
XFM_ROOT = REPO / 'templates' / 'tpl_xfm_build'     # template edges (tpl2tpl, avgsubj) for step 3

problems = neonorm.check_environment(verbose=True)
print('\nready' if not problems else '\nnot ready:\n  - ' + '\n  - '.join(problems))

## 1. The dataset and the results of steps 1-2
The table of `1.quickstart_direct_normalize.ipynb`, section 1; steps 1 and 2 and their reviews (`skullstrip_review.tsv`, `direct_review.tsv` in `DATA_ROOT`) done there.

In [ ]:
visits = pd.read_csv(EXAMPLE / 'participants.tsv', sep='\t', keep_default_na=False)
for c in ('t1', 't2'):  # absolute paths (relative ones are relative to the table)
    visits[c] = [str(EXAMPLE / f) if f else '' for f in visits[c]]
visits['out_dir'] = [str(DATA_ROOT / i) for i in visits['participant_id']]
assert visits['participant_id'].is_unique
DATA_ROOT.mkdir(parents=True, exist_ok=True)
visits

## 2. Step 3: to MNI by each way (`visit_to_mni`)
`visit_to_mni(normalize_json, way, path=(), child=None, xfm_root=None)`

**Input**: `<id>_direct_normalize.json` of step 2; the way; `path`: the BCP templates from the nearest one to BCP60M (then MNI; empty for `direct`); `xfm_root`: the template edges (`templates/tpl_xfm_build/`; computed once from BCP, a new dataset only uses them); `child` (only for `avgsubjl1o`). Ways:
- `direct`: the MNI registration of step 2;
- `tpl2tpl`: the registration to the nearest BCP template (step 2), then the template-to-template registrations along `path` to MNI;
- `avgsubj`: the same chain, with per edge the average of `template a -> visit -> template b` over the BCP (and HCPD) visits of its age window;
- `avgsubjl1o`: avgsubj without the visits of the visit's own child (for checks; for a new dataset the same as avgsubj, as its visits are not in the averages).

| part | function | what | output (`<out_dir>`, `<stem>` = `<id>_<way>[-<path>]_to-MNI152NLin2009cSym`) |
|---|---|---|---|
| 3a | `chain_transforms` / avgsubj edges | all transforms of the way (visit -> nearest template -> ... -> MNI) composed into one displacement field on the MNI 1 mm grid (ANTs `antsApplyTransforms -o [field,1]`) | temporary |
| 3b | `antsApplyTransforms`, `CreateJacobianDeterminantImage` | with that field, one interpolation each: the N4 T1 (linear), the reference cerebellum and the brain mask (nearest neighbour); the log Jacobian determinant of the field (local volume relative to MNI, 0 outside the MNI brain mask) | `<stem>_T1w.nii.gz` (float32), `_cerebellum_dseg.nii.gz`, `_brain_mask.nii.gz` (uint8), `_logjd.nii.gz` (float32) |
| 3c | `mni_metrics` | T1 r with MNI; brain and cerebellum Dice, HD95 and log10(HD95) with MNI; mean and minimum log Jacobian | `metrics` in the json |

**Record**: `<stem>.json`: way, path, the transforms in order, every ANTs command line as run, files, metrics, software; about 14 MB per way. **Resources**: 4 CPUs, 16 GB (ANTs holds all fields of a path, about 7 GB for 20 edges), 1-3 minutes per visit for all ways.

### 2.1 One visit (here in Python), all ways

In [ ]:
V = visits.iloc[0]  # sub-ex01 (infant, 14 edges to MNI); visits.iloc[1]: sub-ex02 (child, 1 edge)
chain = neonorm.adjacent_chain()                     # BCP00M, BCP01M, ..., BCP60M, MNI
start = neonorm.bcp_template(V.age_months)['name']  # the nearest BCP template, registered in step 2
path = chain[chain.index(start):-1]
print(f'{start}: {len(path)} edges to MNI')
nj = Path(V.out_dir) / f'{V.participant_id}_direct_normalize.json'
mni = {way: neonorm.visit_to_mni(nj, way, p, xfm_root=XFM_ROOT)
       for way, p in (('direct', []), ('tpl2tpl', path), ('avgsubj', path))}
display(pd.DataFrame({way: r['metrics'] for way, r in mni.items()}).T)

In [ ]:
# The T1 at MNI by each way, with the MNI cerebellum (red) and the cerebellum of the visit (yellow); one sagittal and
# one axial slice through the cerebellum
tpl = neonorm.mni_template()
load = lambda f: np.squeeze(np.asanyarray(nib.load(f).dataobj))
cereb = load(tpl['cerebellum']) > 0
x, _, z = [int(c) for c in np.argwhere(cereb).mean(0)]
images = {'MNI': (load(tpl['t1']), None)} | {w: (load(r['files']['t1']), load(r['files']['cerebellum']) > 0)
                                             for w, r in mni.items()}
fig, axes = plt.subplots(2, len(images), figsize=(3.2 * len(images), 6.4))
for j, (name, (t1, c)) in enumerate(images.items()):
    for i, cut in enumerate((lambda a: a[x].T, lambda a: a[:, :, z].T)):
        ax = axes[i, j]
        ax.imshow(cut(t1), cmap='gray', origin='lower')
        ax.contour(cut(cereb), [0.5], colors='r', linewidths=0.8)
        if c is not None:
            ax.contour(cut(c), [0.5], colors='y', linewidths=0.8)
        ax.set_axis_off()
    axes[0, j].set_title(name)
fig.tight_layout()

## 3. All visits (Slurm)
`submit_visits_to_mni`: one array task per visit with all ways whose registration is usable (`direct_review.tsv`); per visit the job script and log `<out_dir>/log/<id>_to_mni.sh` / `.out`.

In [ ]:
chain = neonorm.adjacent_chain()
review = pd.read_csv(DATA_ROOT / 'direct_review.tsv', sep='\t')
WAYS = ['direct', 'tpl2tpl', 'avgsubj']


def step3_visit(v):
    """Visit dict of submit_visits_to_mni: the ways whose registration is usable, the path from the nearest template."""
    r = review[review['participant_id'] == v.participant_id]
    ok = lambda role: bool((r.loc[r['role'] == role, 'direct_usable'] == 'yes').any())
    start = r.loc[r['role'] == 'nearest', 'template'].iloc[0]
    path = chain[chain.index(start):-1]
    ways = {w: ([] if w == 'direct' else path) for w in WAYS if ok('MNI' if w == 'direct' else 'nearest')}
    return {'normalize_json': str(Path(v.out_dir) / f'{v.participant_id}_direct_normalize.json'), 'child': v.child,
            'ways': ways}


SUBMIT_STEP3 = False  # True: submit
step3 = [step3_visit(v) for v in visits.itertuples() if v.participant_id in set(review['participant_id'])]
step3 = [s for s in step3 if s['ways']]
print(f"{len(step3)} visits, {sum(len(s['ways']) for s in step3)} visit-ways")
if SUBMIT_STEP3:
    arr = neonorm.submit_visits_to_mni(step3, DATA_ROOT / 'logs' / datetime.now().strftime('step3_%Y%m%d_%H%M%S'),
                                       XFM_ROOT, setup=SETUP, max_running=50)
    print('array', arr['job_id'])

## 4. Using the outputs
- **In MNI**, per visit and way: `_T1w` (intensities; e.g. voxel-based analyses), `_cerebellum_dseg` (the reference cerebellum of the visit), `_brain_mask`, `_logjd` (log Jacobian, the local volume of the visit relative to MNI: negative = smaller; e.g. tensor-based morphometry, longitudinal change between visits of one child).
- **Another image of the visit to MNI** (e.g. a segmentation on the T1 grid) with exactly the same transforms: the `transforms` of the json, in this order, one `antsApplyTransforms` (for `avgsubjl1o` the averages without the child are temporary, `<tmp>`; use `avgsubj` or rerun). Labels with `GenericLabel` or `NearestNeighbor`, intensities with `Linear`.
- **Reproduce** an output: the `commands` of the json are the command lines that made it (`<tmp>`: a temporary folder); with Slurm the same with their full output in `log/<id>_to_mni.out`.

In [ ]:
def to_mni(image, normalize_json, way, out, interpolation='Linear'):
    """Move an image on the grid of the visit's T1 to MNI with the transforms of an existing visit_to_mni json."""
    nj = Path(normalize_json)
    prefix = json.loads(nj.read_text())['inputs']['prefix']
    jsons = sorted(nj.parent.glob(f'{prefix}_{way}*_to-{neonorm.MNI_NAME}.json'))
    j = json.loads(jsons[0].read_text())
    if any(t.startswith('<tmp>') for t in j['transforms']):
        raise ValueError(f'{jsons[0].name}: temporary fields (avgsubjl1o)')
    neonorm.run_ants('antsApplyTransforms', '-d', 3, '-i', image, '-r', neonorm.mni_template()['t1'], '-o', out,
                     '-n', interpolation, *[x for t in j['transforms'] for x in ('-t', t)])
    return out


# e.g. the brain mask of step 1 to MNI by tpl2tpl:
# v = visits.iloc[0]
# to_mni(Path(v.out_dir) / f'{v.participant_id}_T1w_brain_mask.nii.gz', Path(v.out_dir) / f'{v.participant_id}_direct_normalize.json',
#        'tpl2tpl', Path(v.out_dir) / f'{v.participant_id}_tpl2tpl_brain_mask_in_MNI.nii.gz', 'NearestNeighbor')

## 5. Which way for which age (open)
The evaluation (`Notebook/5_evaluation.ipynb`: cerebellum and brain overlap with MNI, T1 r, longitudinal logJD, by age; BCP and HCPD) decides from which age direct normalisation to MNI is good enough, and below it whether tpl2tpl or avgsubj (and with how many edges, notebook 4, 4.4). Until then a new dataset runs all three ways; the choice then only selects files, nothing has to run again. This section will state the rule once it is known.